In [ ]:
from google.cloud import storage
import pandas as pd
from utils.config import Config
import gcsfs
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
BUCKET_NAME = Config.RAW_DATA_BUCKET

storage_client = storage.Client()
blobs = storage_client.list_blobs(BUCKET_NAME)

print(f"Files in {BUCKET_NAME}:")
for blob in blobs:
    print(f"- {blob.name}")

In [ ]:
fs = gcsfs.GCSFileSystem(project=Config.PROJECT_ID)

files = fs.ls('retail-analysis-project-492114-liquor-raw/raw_data/liquor_sales/')
print(files)

In [ ]:
df = pd.concat([pd.read_parquet(f"gs://{f}") for f in files])

In [ ]:
df.info()
df.head()

In [ ]:
print(df['county'].value_counts().head(20))

print(f"Unique counties: {df['county'].nunique()}")

print(f"Null counties: {df['county'].isna().sum()}")

In [ ]:
# Do rows with missing counties have city data
missing_county = df[df['county'].isna()]
print("Cities with missing county data:")
print(missing_county['city'].value_counts())
print("Address with missing county data:")
print(missing_county['address'].value_counts())
print("zipcode with missing county data:")
print(missing_county['zipcode'].value_counts())

In [ ]:
# List of columns that should be numbers
numeric_cols = [
    'state_bottle_cost', 
    'state_bottle_retail', 
    'sale_bottles', 
    'sale_dollars', 
    'sale_liters'
]

# Convert strings to numeric, turning errors (if any) into NaN
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Create transaction_type column
df['transaction_type'] = 'SALE'
df.loc[df['sale_dollars'] <= 0, 'transaction_type'] = 'RETURN'

# Create the boolean flag
df['is_return'] = df['sale_dollars'] <= 0

In [ ]:
sns.set_theme(style="whitegrid")

cols_to_corr = ['state_bottle_cost', 'state_bottle_retail', 'sale_bottles', 'sale_dollars', 'sale_liters']
correlation_data = df[cols_to_corr].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(correlation_data, annot=True, fmt=".2f", cmap='RdBu_r', center=0)
plt.title('Iowa Liquor Sales: Feature Correlation')
plt.show()

In [ ]:
# Correlation for actual Sales
sales_corr = df[df['transaction_type'] == 'SALE'][cols_to_corr].corr()

# Correlation for Returns
returns_corr = df[df['transaction_type'] == 'RETURN'][cols_to_corr].corr()

# Difference in Dollars vs Liters
print(f"Sales Correlation (Liters/Dollars): {sales_corr.loc['sale_liters', 'sale_dollars']:.4f}")
print(f"Returns Correlation (Liters/Dollars): {returns_corr.loc['sale_liters', 'sale_dollars']:.4f}")

In [ ]:
category_list = sorted(df['category_name'].dropna().unique().tolist())

for category in category_list:
    print(f"'{category}'")

In [ ]:
vendor_list = sorted(df['vendor_name'].dropna().unique().tolist())

for vendor in vendor_list:
    print(f"'{vendor}'")

In [ ]:
vodka_variations = df[df['category_name'].str.contains('VODKA', case=False, na=False)]['category_name'].unique()
print("\n--- All variations of 'Vodka' in the dataset ---")
print(vodka_variations)